# L3 — Orchestrator Harness

## What you'll learn

- How to create the orchestrator, an AgentCore Harness that finds the specialist agents in the Registry and delegates to them
- How to give it short-term and long-term memory
- How to turn on observability for the Harness, the memory and the agents
- How to test the orchestrator end to end and stream its answer

## Overview

The orchestrator is the top-level agent in AnyCompany's customer support system. It holds the conversation with the customer, finds the specialist agents in the Registry (the Order and Refund agents, and the optional Research Agent), and delegates each request to the right one.

- **Model.** The orchestrator model you chose in L2, on Amazon Bedrock or Baseten. It is reached through LiteLLM, like the specialist agents' models.
- **Memory.** Short-term chat history, plus long-term summaries, preferences and facts for each customer.
- **Instructions.** A system prompt that says when to delegate, sets the refund rules and explains how to handle upset customers. You can read and edit it in `prompts/orchestrator_system_prompt.md`.

## Architecture

![Orchestration layer: the AgentCore Harness with its memory discovers the Order, Refund and Research agents through the Registry; the agents call tools through the AgentCore Gateway, which reaches the order tools and DynamoDB, the refund tools and the knowledge base (Bedrock or Couchbase), and one web-search provider.](images/l3-orchestration-architecture.png)

This notebook creates the Harness shown on the left of the diagram, together with its memory. The agents and the Gateway were set up in the earlier notebooks.

## Steps

0. Set up: install the packages and load the libraries
1. Load the workshop configuration
2. Create the memory
3. Create the Harness (model and system prompt)
4. Test the Harness
5. Save the IDs
6. Turn on observability (choice made in the Partner Selection UI)

## License details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- L2 ([`1_pluggable_inference_layer.ipynb`](../l2-inference/1_pluggable_inference_layer.ipynb)) is complete.
- [`1_gateway_tools.ipynb`](1_gateway_tools.ipynb) and [`2_specialist_agents.ipynb`](2_specialist_agents.ipynb) are complete.

> **Note:** you can run [`1_sensitive_data_masking.ipynb`](../l4-security/1_sensitive_data_masking.ipynb) and [`1_decision_models.ipynb`](../l5-application/1_decision_models.ipynb) in parallel with this notebook.

## Step 0: Set up

### 0.1 Install the packages

Install the AWS SDK and the AgentCore library. Run this once, at the start.

In [ ]:
%pip install --quiet --upgrade boto3 bedrock-agentcore==1.14.0

### 0.2 Load the libraries

Import the libraries and set up the AWS connections.

In [ ]:
import boto3
import json
import os
import uuid
import time

# --- Configuration ---
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1") # Change to your preferred region

# Clients
from botocore.config import Config
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
agentcore_runtime = boto3.client("bedrock-agentcore", region_name=REGION,
                                  config=Config(read_timeout=300))
sts = boto3.client("sts", region_name=REGION)

ACCOUNT_ID = sts.get_caller_identity()["Account"]
print(f"Account: {ACCOUNT_ID}")
print(f"Region:  {REGION}")

## Step 1: Load the workshop configuration

### 1.1 Load the Registry and Gateway IDs

Read the Registry and Gateway IDs and the role the Harness runs as, which the workshop saved earlier.

In [ ]:
SSM_PREFIX = "/anycompany/agentcore"
ssm = boto3.client("ssm", region_name=REGION)

def get_ssm(name: str) -> str:
    """Retrieve a single SSM parameter value."""
    return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]

REGISTRY_ID = get_ssm(f"{SSM_PREFIX}/registry_id")
REGISTRY_ARN = get_ssm(f"{SSM_PREFIX}/registry_arn")
GATEWAY_ID = get_ssm(f"{SSM_PREFIX}/gateway_id")
GATEWAY_ARN = get_ssm(f"{SSM_PREFIX}/gateway_arn")
# The Harness execution role is pre-provisioned by the L3 orchestration stack.
EXECUTION_ROLE_ARN = get_ssm(f"{SSM_PREFIX}/harness_role_arn")

print(f"Registry ID:  {REGISTRY_ID}")
print(f"Registry ARN: {REGISTRY_ARN}")
print(f"Gateway ID:   {GATEWAY_ID}")
print(f"Gateway ARN:  {GATEWAY_ARN}")
print(f"Harness role: {EXECUTION_ROLE_ARN}")

# Observability provider (selected in the Partner Selection UI) — branch CloudWatch native vs Datadog OTEL-direct.
from helpers.observability import get_observability_provider, datadog_otel_env, mask_secret
OBS_PROVIDER = get_observability_provider(ssm, SSM_PREFIX)
print(f"Observability: {OBS_PROVIDER}")

## Step 2: Create the memory

The memory keeps each customer's conversations. Short-term memory, the recent conversation, is automatic. Long-term memory needs three strategies, which this step sets up: summaries of past sessions, the customer's preferences, and facts the customer has shared.

### 2.1 Create the memory

Create the memory, kept separately for each customer.

In [ ]:
MEMORY_NAME = "anycompany_agent_memory"
print(f"Creating memory: {MEMORY_NAME}")

try:
    memory_response = agentcore_control.create_memory(
        name=MEMORY_NAME,
        description="long-term memory for the AnyCompany orchestrator harness.",
        memoryExecutionRoleArn=EXECUTION_ROLE_ARN,
        eventExpiryDuration=30,  # Events expire after 30 days
        memoryStrategies=[
            {
                "summaryMemoryStrategy": {
                    "name": "session_summary",
                    "description": "Summarizes each conversation session",
                    # `namespaceTemplates` (not the legacy `namespaces`) supports
                    # variable substitution. {actorId}/{sessionId} ensures each
                    # user's session summaries are isolated from other users'.
                    "namespaceTemplates": ["summaries/{actorId}/{sessionId}"]
                }
            },
            {
                "userPreferenceMemoryStrategy": {
                    "name": "user_preferences",
                    "description": "Extracts stated user preferences",
                    # Per-user preference scoping. Without {actorId} every
                    # user's preferences would collide in a flat namespace.
                    "namespaceTemplates": ["preferences/{actorId}"]
                }
            },
            {
                "semanticMemoryStrategy": {
                    "name": "semantic_facts",
                    "description": "Extracts factual knowledge for semantic retrieval",
                    "namespaceTemplates": ["facts/{actorId}"]
                }
            }
        ]
    )

    MEMORY_ID = memory_response["memory"]["id"]
    MEMORY_ARN = memory_response["memory"]["arn"]
    print(f"Created memory: {MEMORY_NAME}")
    print(f"  ID:  {MEMORY_ID}")
    print(f"  ARN: {MEMORY_ARN}")

except (agentcore_control.exceptions.ConflictException, agentcore_control.exceptions.ValidationException):
    print(f"Memory '{MEMORY_NAME}' already exists. Looking up...")
    memories = agentcore_control.list_memories()
    for mem in memories.get("memories", []):
        # list_memories doesn't return name, so get each one
        detail = agentcore_control.get_memory(memoryId=mem["id"])
        if detail["memory"].get("name") == MEMORY_NAME:
            MEMORY_ID = detail["memory"]["id"]
            MEMORY_ARN = detail["memory"]["arn"]
            print(f"  ID:  {MEMORY_ID}")
            print(f"  ARN: {MEMORY_ARN}")
            break
    else:
        raise RuntimeError(f"Could not find memory with name '{MEMORY_NAME}'")

### 2.2 Wait for the memory

Wait until the memory is ready.

In [ ]:
# Wait for memory to be ready
import time as _time

print("Waiting for memory to be ready...")
_mem_start = _time.time()
_poll = 0

while True:
    m = agentcore_control.get_memory(memoryId=MEMORY_ID)
    status = m["memory"]["status"]
    _elapsed = _time.time() - _mem_start
    _poll += 1
    if status == "ACTIVE":
        print(f"Memory is ACTIVE after {_elapsed:.1f}s ({_poll} poll(s))")
        break
    elif status == "FAILED":
        reason = m["memory"].get("failureReason", "unknown")
        raise RuntimeError(f"Memory creation failed: {reason}")
    print(f"  [{_elapsed:5.1f}s] Status: {status} — waiting... (poll #{_poll})")
    time.sleep(5)

print(f"\nMemory ID:  {MEMORY_ID}")
print(f"Memory ARN: {MEMORY_ARN}")

## Step 3: Create the Harness

### 3.1 Load the system prompt

The system prompt that guides the orchestrator is in [`prompts/orchestrator_system_prompt.md`](prompts/orchestrator_system_prompt.md): how to tell statements from requests, how to find and call the specialist agents, the refund-eligibility rules, product research, upset customers and escalation, memory, response style and scope. Open the file to read or edit the rules, then re-run this cell and Steps 3.2 and 3.3 to apply your changes.

In [ ]:
from pathlib import Path

# The orchestrator's instructions live in a file so they are easy to read and edit.
SYSTEM_PROMPT = Path("prompts/orchestrator_system_prompt.md").read_text()

print("System prompt loaded from prompts/orchestrator_system_prompt.md")
print(f"Length: {len(SYSTEM_PROMPT)} characters")

### 3.2 Choose the model

L2 saved your model choice, but it didn't configure the Harness. The Harness is a managed service, so it needs the model, and for a partner model the API key, in its own settings. This cell reads your L2 choice and turns it into those settings. The Harness calls its model through LiteLLM, like the specialist agents:

- **Amazon Bedrock:** the Harness signs in with its own role, so no API key is needed.
- **Baseten:** the API key you saved in the Partner Selection UI is stored securely in AgentCore Identity and passed to the Harness, so the key never appears in the Harness's configuration.

In [ ]:
from botocore.exceptions import ClientError

HARNESS_KEY_PROVIDER = "anycompany-orchestrator-model-key"   # the Harness role can read only this key

def _ssm_optional(name):
    try:
        return get_ssm(name)
    except ssm.exceptions.ParameterNotFound:
        return None

harness_model_id = get_ssm(f"{SSM_PREFIX}/harness_model_id")
if "/" not in harness_model_id:
    # An earlier L2 run published a bare Bedrock id; LiteLLM needs the provider prefix.
    harness_model_id = f"bedrock/{harness_model_id}"

# No temperature: newer models (Claude Opus 5, Sonnet 5) reject it, so the model uses its default.
lite_llm_config = {"modelId": harness_model_id, "maxTokens": 4096}
if harness_model_id.startswith("bedrock/"):
    print(f"Model: {harness_model_id} (LiteLLM -> Amazon Bedrock, execution role, API key not needed)")
else:
    # Only providers with a custom endpoint publish one; baseten/... does not need it.
    harness_api_base = _ssm_optional(f"{SSM_PREFIX}/harness_api_base")
    _api_key = _ssm_optional(f"{SSM_PREFIX}/config/inference/orchestrator/api_key")
    if not _api_key:
        raise RuntimeError("A partner orchestrator model is selected but no API key is saved. Save it in "
                           "the Partner Selection UI (Inference -> Orchestrator Agent), then re-run this cell.")
    try:
        _provider = agentcore_control.create_api_key_credential_provider(
            name=HARNESS_KEY_PROVIDER, apiKey=_api_key)
        print(f"Stored the API key in AgentCore Identity: {HARNESS_KEY_PROVIDER}")
    except ClientError as e:
        _code = e.response["Error"]["Code"]
        if not (_code == "ConflictException" or
                (_code == "ValidationException" and "already exists" in str(e).lower())):
            raise
        # Re-running (or the key was rotated in the UI): update the provider in place.
        _provider = agentcore_control.update_api_key_credential_provider(
            name=HARNESS_KEY_PROVIDER, apiKey=_api_key)
        print(f"Updated the API key in AgentCore Identity: {HARNESS_KEY_PROVIDER}")
    finally:
        del _api_key   # never keep the raw key in the notebook namespace
    lite_llm_config["apiKeyArn"] = _provider["credentialProviderArn"]
    if harness_api_base:
        lite_llm_config["apiBase"] = harness_api_base
    print(f"Model: {harness_model_id} (LiteLLM -> {harness_api_base or harness_model_id.split('/')[0]}, key from AgentCore Identity)")

HARNESS_MODEL = {"liteLlmModelConfig": lite_llm_config}

### 3.3 Create the Harness

Create the Harness with that model, the system prompt, the memory, the Registry tools and Code Interpreter. If the Harness already exists, it is updated, so re-run Steps 3.2 and 3.3 after you change the model in L2.

In [ ]:
print("Creating AgentCore Harness...")
HARNESS_NAME = "anycompany_orchestrator"
# Datadog only: send the Harness traces to Datadog (logs stay in CloudWatch).
if OBS_PROVIDER == "datadog":
    _harness_otel_env = datadog_otel_env(ssm, "anycompany-orchestrator", SSM_PREFIX)
    # The managed Harness needs this header for its traces to appear in Datadog Agent Observability
    # (the dd_llmobs_enabled attribute the specialists use doesn't work here).
    _dd_key = _harness_otel_env["OTEL_EXPORTER_OTLP_TRACES_HEADERS"].split("dd-api-key=", 1)[-1].split(",")[0]
    _harness_otel_env["OTEL_EXPORTER_OTLP_TRACES_HEADERS"] = f"dd-api-key={_dd_key},compute_stats=true,dd-otlp-source=llmobs"
    _harness_otel_env["OTEL_RESOURCE_ATTRIBUTES"] = "deployment.environment=demo,deployment.environment.name=demo"
    _masked_key = mask_secret(_dd_key)
    print(f"Datadog OTEL-direct tracing enabled for Harness (dd-api-key={_masked_key}).")
else:
    _harness_otel_env = {}

try:
    harness_response = agentcore_control.create_harness(
        harnessName=HARNESS_NAME,
        executionRoleArn=EXECUTION_ROLE_ARN,

        # Model: liteLlmModelConfig built in the previous cell (Bedrock or a partner provider)
        model=HARNESS_MODEL,

        # System prompt
        systemPrompt=[{"text": SYSTEM_PROMPT}],

        # Tools — gateway (for registry discovery) + code interpreter
        tools=[
            {"type": "agentcore_gateway", "name": "registry-tools", "config": {"agentCoreGateway": {"gatewayArn": get_ssm(f"{SSM_PREFIX}/harness_gateway_arn")}}},
            {"type": "agentcore_code_interpreter", "name": "code_interpreter", "config": {"agentCoreCodeInterpreter": {}}},
        ],

        allowedTools=["*"],

        # Memory
        memory={
            "agentCoreMemoryConfiguration": {
                "arn": MEMORY_ARN,
                "messagesCount": 20
            }
        },

        # Execution limits
        maxIterations=20,
        maxTokens=16384,
        timeoutSeconds=300,

        tags={
            "project": "anycompany-workshop",
            "layer": "l3-orchestration",
            "role": "orchestrator"
        },
        # Datadog OTLP-direct tracing (traces only; logs & metrics stay in CloudWatch).
        **({"environmentVariables": _harness_otel_env} if _harness_otel_env else {}),
    )

    harness = harness_response["harness"]
    HARNESS_ID = harness["harnessId"]
    HARNESS_ARN = harness["arn"]
    print(f"\nHarness created!")
    print(f"  Name:   {harness['harnessName']}")
    print(f"  ID:     {HARNESS_ID}")
    print(f"  ARN:    {HARNESS_ARN}")
    print(f"  Status: {harness['status']}")

except agentcore_control.exceptions.ConflictException:
    print(f"Harness '{HARNESS_NAME}' already exists. Updating with new system prompt...")
    harnesses = agentcore_control.list_harnesses()
    for h in harnesses.get("harnesses", []):
        if h.get("harnessName") == HARNESS_NAME:
            HARNESS_ID = h["harnessId"]
            HARNESS_ARN = h["arn"]
            break
    else:
        raise RuntimeError(f"Could not find harness with name '{HARNESS_NAME}'")

    agentcore_control.update_harness(
        harnessId=HARNESS_ID,
        executionRoleArn=EXECUTION_ROLE_ARN,
        model=HARNESS_MODEL,
        systemPrompt=[{"text": SYSTEM_PROMPT}],
        tools=[
            {"type": "agentcore_gateway", "name": "registry-tools", "config": {"agentCoreGateway": {"gatewayArn": get_ssm(f"{SSM_PREFIX}/harness_gateway_arn")}}},
            {"type": "agentcore_code_interpreter", "name": "code_interpreter", "config": {"agentCoreCodeInterpreter": {}}}
        ],
        allowedTools=["*"],
        memory={"optionalValue": {"agentCoreMemoryConfiguration": {"arn": MEMORY_ARN, "messagesCount": 20}}},
        maxIterations=20,
        maxTokens=16384,
        timeoutSeconds=300,
        # Datadog OTLP-direct tracing (traces only; logs & metrics stay in CloudWatch).
        **({"environmentVariables": _harness_otel_env} if _harness_otel_env else {}),
    )
    print(f"  Harness updated with new system prompt.")
    print(f"  ID:  {HARNESS_ID}")
    print(f"  ARN: {HARNESS_ARN}")

### 3.4 Wait for the Harness

Wait until the Harness is ready.

In [ ]:
print("Waiting for harness to be ready...")
for _ in range(30):
    harness_info = agentcore_control.get_harness(harnessId=HARNESS_ID)
    status = harness_info["harness"]["status"]
    if status == "READY":
        print(f"Harness is READY!")
        break
    elif "FAIL" in status:
        reason = harness_info["harness"].get("failureReason", "unknown")
        raise RuntimeError(f"Harness creation failed: {status} — {reason}")
    print(f"  Status: {status} — waiting 10s...")
    time.sleep(10)
else:
    raise TimeoutError("Harness did not become ready within 5 minutes.")

print(f"\nHarness ID:  {HARNESS_ID}")
print(f"Harness ARN: {HARNESS_ARN}")

## Step 4: Test the Harness

### 4.1 Ask a refund question

Send the Harness a sample refund question and watch the answer stream in, to confirm that orchestration works end to end.

In [ ]:
session_id = str(uuid.uuid4()).upper()
# actorId keeps long-term memory separate per customer. The chatbot uses the signed-in
# customer's ID; this test uses a fixed Gold-tier customer.
actor_id = "CUST-789"
print(f"Session ID: {session_id}")
print(f"Actor ID:   {actor_id}\n")

response = agentcore_runtime.invoke_harness(
    harnessArn=HARNESS_ARN,
    runtimeSessionId=session_id,
    actorId=actor_id,
    messages=[{"role": "user", "content": [{"text": f"[Session: {session_id}]\n\nHi! I want to know my refund for ORD-10001. Customer ID is CUST-789."}]}],
)

# Process the streaming response
for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

## Step 5: Save the IDs

Save the Harness and memory IDs for the observability notebooks in L6.

In [ ]:
parameters = {
    f"{SSM_PREFIX}/harness_id": HARNESS_ID,
    f"{SSM_PREFIX}/harness_arn": HARNESS_ARN,
    f"{SSM_PREFIX}/harness_name": HARNESS_NAME,
    f"{SSM_PREFIX}/memory_id": MEMORY_ID,
    f"{SSM_PREFIX}/memory_arn": MEMORY_ARN,
    f"{SSM_PREFIX}/memory_name": MEMORY_NAME,
}

print(f"Saving {len(parameters)} parameters to SSM under '{SSM_PREFIX}/'\n")

for name, value in parameters.items():
    ssm.put_parameter(
        Name=name,
        Value=value,
        Type="String",
        Overwrite=True,
    )
    print(f"  ✓ {name} = {value}")

print(f"\nAll parameters saved.")

## Step 6: Turn on observability

This step turns on observability for all the L3 resources at once: the Order and Refund agents, the orchestrator's memory and the Gateway. It uses the observability provider you chose in the Partner Selection UI:

- **CloudWatch:** traces and logs go to CloudWatch.
- **Datadog:** logs go to CloudWatch, and traces go to Datadog through the exporters that were set up when the agents were deployed.

It is safe to re-run.

In [ ]:
if OBS_PROVIDER == "datadog":
    from helpers import datadog as obs
else:
    from helpers import cloudwatch as obs

logs_client = boto3.client('logs', region_name=REGION)

RESOURCES = [
    ('runtime', 'order_agent_runtime_arn'),
    ('runtime', 'refund_agent_runtime_arn'),
    ('memory',  'memory_arn'),
    ('gateway', 'gateway_arn'),
]

for kind, ssm_key in RESOURCES:
    arn = get_ssm(f'{SSM_PREFIX}/{ssm_key}')
    resource_id = arn.split('/')[-1]
    print(f'\n--- {kind}: {resource_id} ---')
    obs.enable(logs_client, kind, arn, resource_id, ACCOUNT_ID)

## Summary

This notebook created the orchestrator: a Harness that runs the model you chose in L2, with short-term and long-term memory. It finds the specialist agents through the Registry and delegates each request to the right one. Observability is turned on for the agents, the memory and the Gateway.

**Next:** [`1_sensitive_data_masking.ipynb`](../l4-security/1_sensitive_data_masking.ipynb). Protect customer data in the agents' answers.

---
## Cleanup

Deletes what this notebook created: the Harness, the memory, the log delivery wiring for the memory and the Gateway, and the saved IDs. The Harness's IAM role belongs to the workshop stack and is removed when it's deleted.

Run this **before** the cleanup in [`1_gateway_tools.ipynb`](1_gateway_tools.ipynb), because this notebook owns the Gateway's log delivery wiring.

Uncomment and run to delete all resources created in this notebook.

In [ ]:
## === CLEANUP: Delete all resources created in this notebook ===
## ⚠️ Uncomment and run this cell only after you are done with ALL notebooks.
## NOTE: Run this BEFORE the 1_gateway_tools.ipynb cleanup, since this
##       notebook owns the gateway log delivery wiring (the gateway itself is
##       pre-provisioned by the CloudFormation stack).

# import boto3, os, time
#
## Note: In production, implement content filtering if responses may contain sensitive user data from memory or tool results.
## Content filtering should apply redaction or field selection before displaying responses to users.
## PII in tool responses is masked by the L4 guardrail interceptor (l4-security/1_sensitive_data_masking.ipynb).
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
#
# HARNESS_NAME = "anycompany_orchestrator"
# HARNESS_KEY_PROVIDER = "anycompany-orchestrator-model-key"   # Step 3.2, partner models only
# MEMORY_NAME  = "anycompany_agent_memory"
#
# ssm               = boto3.client("ssm", region_name=REGION)
# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
# logs_client       = boto3.client("logs", region_name=REGION)
#
# def _ssm_get(key):
#     try:
#         return ssm.get_parameter(Name=f"{SSM_PREFIX}/{key}")["Parameter"]["Value"]
#     except ssm.exceptions.ParameterNotFound:
#         return None
#
# # Resolve IDs/ARNs from SSM, falling back to lookup-by-name (idempotent).
# HARNESS_ID  = _ssm_get("harness_id")
# HARNESS_ARN = _ssm_get("harness_arn")
# MEMORY_ID   = _ssm_get("memory_id")
# MEMORY_ARN  = _ssm_get("memory_arn")
# GATEWAY_ARN = _ssm_get("gateway_arn")
# GATEWAY_ID  = _ssm_get("gateway_id")
#
# if not (HARNESS_ID and HARNESS_ARN):
#     try:
#         for h in agentcore_control.list_harnesses().get("harnesses", []):
#             if h.get("harnessName") == HARNESS_NAME:
#                 HARNESS_ID  = h["harnessId"]
#                 HARNESS_ARN = h["arn"]
#                 break
#     except Exception as e:
#         print(f"List harnesses: {e}")
#
# if not (MEMORY_ID and MEMORY_ARN):
#     try:
#         for mem in agentcore_control.list_memories().get("memories", []):
#             try:
#                 detail = agentcore_control.get_memory(memoryId=mem["id"])
#                 if detail["memory"].get("name") == MEMORY_NAME:
#                     MEMORY_ID  = detail["memory"]["id"]
#                     MEMORY_ARN = detail["memory"]["arn"]
#                     break
#             except Exception:
#                 continue
#     except Exception as e:
#         print(f"List memories: {e}")
#
# # ── Phase 1: Delete Harness (must precede Memory delete) ────────────────────
# if HARNESS_ID:
#     try:
#         agentcore_control.delete_harness(harnessId=HARNESS_ID)
#         print(f"Initiated delete of Harness: {HARNESS_ID}")
#         deadline = time.time() + 300  # 5 min
#         while time.time() < deadline:
#             try:
#                 agentcore_control.get_harness(harnessId=HARNESS_ID)
#                 time.sleep(10)
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print("Harness fully deleted.")
#                 break
#         else:
#             print("Timed out waiting for Harness to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print(f"Harness {HARNESS_ID} already deleted.")
#     except Exception as e:
#         print(f"Harness delete: {e}")
# else:
#     print(f"Harness {HARNESS_NAME} not found (already deleted).")
#
# # Helper: drain log delivery for a resource (never deletes the shared XRAY destination).
# from helpers import disable_log_delivery
# def _drain_log_delivery(resource_arn, dest_name_prefix):
#     disable_log_delivery(logs_client, resource_arn, dest_name_prefix)
#
# # ── Phase 2: Drain Memory CloudWatch log delivery ───────────────────────────
# if MEMORY_ARN:
#     _drain_log_delivery(MEMORY_ARN, dest_name_prefix="mem-")
# else:
#     print("Memory ARN not resolved; skipping memory log delivery cleanup.")
#
# # ── Phase 3: Drain Gateway CloudWatch log delivery (wiring only) ────────────
# if GATEWAY_ARN:
#     _drain_log_delivery(GATEWAY_ARN, dest_name_prefix="gw-")
# else:
#     print("Gateway ARN not resolved; skipping gateway log delivery cleanup.")
#
# # ── Phase 4: Delete Memory and poll until gone ─────────────────────────────
# if MEMORY_ID:
#     try:
#         agentcore_control.delete_memory(memoryId=MEMORY_ID)
#         print(f"Initiated delete of Memory: {MEMORY_ID}")
#         deadline = time.time() + 300
#         while time.time() < deadline:
#             try:
#                 agentcore_control.get_memory(memoryId=MEMORY_ID)
#                 time.sleep(10)
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print("Memory fully deleted.")
#                 break
#         else:
#             print("Timed out waiting for Memory to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print(f"Memory {MEMORY_ID} already deleted.")
#     except Exception as e:
#         print(f"Memory delete: {e}")
# else:
#     print(f"Memory {MEMORY_NAME} not found (already deleted).")
#
# # ── Phase 5: Delete CloudWatch log groups ──────────────────────────────────
# log_groups = []
# if MEMORY_ID:
#     log_groups.append(f"/aws/vendedlogs/bedrock-agentcore/memory/APPLICATION_LOGS/{MEMORY_ID}")
# if GATEWAY_ID:
#     log_groups.append(f"/aws/vendedlogs/bedrock-agentcore/gateway/APPLICATION_LOGS/{GATEWAY_ID}")
# for lg in log_groups:
#     try:
#         logs_client.delete_log_group(logGroupName=lg)
#         print(f"Deleted log group: {lg}")
#     except logs_client.exceptions.ResourceNotFoundException:
#         pass
#     except Exception as e:
#         print(f"Log group delete ({lg}): {e}")
#
# # The Harness execution role is owned by the CloudFormation stack (not deleted here).
#
# # ── Phase 6: Delete SSM parameters ─────────────────────────────────────────
# for param in [
#     "harness_id", "harness_arn", "harness_name",
#     "memory_id", "memory_arn", "memory_name",
# ]:
#     try:
#         ssm.delete_parameter(Name=f"{SSM_PREFIX}/{param}")
#         print(f"Deleted SSM parameter: {SSM_PREFIX}/{param}")
#     except ssm.exceptions.ParameterNotFound:
#         pass
#     except Exception as e:
#         print(f"SSM delete ({param}): {e}")
#
# # Remove the orchestrator model API key from AgentCore Identity (Step 3.2, partner models only).
# try:
#     agentcore_control.delete_api_key_credential_provider(name=HARNESS_KEY_PROVIDER)
#     print(f"Deleted API-key credential provider: {HARNESS_KEY_PROVIDER}")
# except agentcore_control.exceptions.ResourceNotFoundException:
#     pass
# except Exception as e:
#     print(f"API-key credential provider delete: {e}")
#
# print("\n✅ All Orchestrator resources cleaned up.")